# 04a — Fixed-log: The Fleet App

Turn a monitor into a small app on your machine: a layout you choose, alerts, a schedule,
and buttons to add files and run. Uses the complete model from `02a or 02b` (or the
ready-made sample model in `sample_data/models/`) and the 13 weekly sample files.

See `04_fleet_app_guide.ipynb` for what each step does and why.

## Setup

In [ ]:
import glob, json, os, shutil, sqlite3, subprocess, sys, time, urllib.request, webbrowser
from pathlib import Path

import pandas as pd

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

import amygda_ops_risk_score
from amygda_ops_risk_score import OpsRiskClient, helpers
from amygda_ops_risk_score.monitor import Monitor, SpecError, validate_spec

# The `amygda-risk` command. If this notebook's kernel can't find it on PATH (its
# environment isn't activated), the same command is run through this kernel's Python.
AMYGDA_RISK = shutil.which("amygda-risk") or f'"{sys.executable}" -m amygda_ops_risk_score.monitor.cli'

print("amygda_ops_risk_score version:", amygda_ops_risk_score.__version__)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

API_KEY = os.environ.get("AMYGDA_SDK_API_KEY") or "paste_your_api_key_here"   # ← paste your key here

# The monitor, the `amygda-risk` command and the app read the key from AMYGDA_API_KEY.
# It is never written to the monitor folder.
os.environ["AMYGDA_API_KEY"] = API_KEY

client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
# Yours from 02a or 02b if you ran it, otherwise the ready-made sample model.
MODEL_ZIPS = sorted(glob.glob("outputs/fixed_log/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "sample_data/models/fixed_log_complete_model.zip"
TRAINING   = "sample_data/fixed_log_training.csv"     # earlier logs, used as history
BATCHES    = Path("sample_data/batches/fixed_log")    # week_01.csv … week_13.csv
MON_DIR    = Path("outputs/fixed_log_fleet_app")
MON_NAME   = "example-fleet-app"
SPEC_FILE  = Path("outputs/fixed_log_fleet_app_spec.json")
PORT       = 8050

week = {int(p.stem[-2:]): p for p in sorted(BATCHES.glob("week_*.csv"))}
print("Model:", MODEL)

# Start clean: remove the folder and deployment from an earlier run of this example
for d in client.list_deployments(API_KEY):
    if d["name"] == MON_NAME:
        client.delete_deployment(API_KEY, d["deployment_id"])
shutil.rmtree(MON_DIR, ignore_errors=True)

## Create the monitor, with a database for storage

`--storage sqlite` keeps the scores, runs and alerts in one file, `monitor.db`. Leave it
out to keep them as ordinary files.

In [ ]:
!{AMYGDA_RISK} init {MON_DIR} --model {MODEL} --name {MON_NAME} \
    --history {TRAINING} --fleet-name "Baggage sortation stations" --storage sqlite

## Choose what the app shows

The spec picks the tabs, panels, header figures, look, alert rules and schedule.
`amygda-risk panels` lists everything that can be chosen.

In [ ]:
spec = {
    "title": "Sortation stations",
    "rank_by": "peak",
    "theme": {"mode": "auto", "accent": "#0b6e4f"},
    "header": {"tiles": ["open_alerts", "assets_high_now", "highest", "last_run"]},
    "alerts": {"rules": [
        {"type": "enters_band", "band": "watch"},
        {"type": "rises_by", "points": 8},
        {"type": "run_failed", "runs": 1},
    ]},
    "schedule": {"every": "day", "at": "06:00"},
    "tabs": [
        {"title": "Fleet",  "panels": ["since_last_run", "fleet_systems",
                                        {"type": "ranking", "top": 10}]},
        {"title": "Alerts", "panels": ["alerts"]},
        {"title": "Assets", "panels": ["asset_trend", "asset_systems", "asset_drivers"]},
        {"title": "Health", "panels": ["run_health", "run_table", "known_gaps"]},
    ],
}
SPEC_FILE.write_text(json.dumps(spec, indent=1))
!{AMYGDA_RISK} design {MON_DIR} --spec {SPEC_FILE}

## Score the first weeks

In [ ]:
for n in range(1, 7):
    !{AMYGDA_RISK} run {MON_DIR} --input {week[n]} --no-dashboard

In [ ]:
!{AMYGDA_RISK} alerts {MON_DIR}

## Start the app

In a terminal this is `amygda-risk app outputs/fixed_log_fleet_app`. Here it is started in
the background. The page has **Add files**, **Run now** and **Schedule** in a bar at the top.

In [ ]:
app = subprocess.Popen([sys.executable, "-m", "amygda_ops_risk_score.monitor.cli",
                        "app", str(MON_DIR), "--port", str(PORT), "--no-open"])
time.sleep(3)
_ = webbrowser.open(f"http://127.0.0.1:{PORT}/")

Add a file and run it, as the **Add files** and **Run now** buttons do:

In [ ]:
APP_URL = f"http://127.0.0.1:{PORT}"

def app_call(method, path, body=None):
    """Call the app the way its page does."""
    data = body if isinstance(body, bytes) else (json.dumps(body).encode() if body is not None else None)
    req = urllib.request.Request(APP_URL + path, data=data, method=method,
                                 headers={"X-Amygda-App": "1"})
    with urllib.request.urlopen(req, timeout=60) as res:
        return json.loads(res.read())

def wait_for_run():
    """Wait until the app has no run in progress, then return its state."""
    while True:
        state = app_call("GET", "/api/state")
        if not state["running"]:
            return state
        time.sleep(1)

app_call("PUT", f"/api/inbox/{week[7].name}", week[7].read_bytes())
app_call("POST", "/api/run")
state = wait_for_run()
print("Last run:", state["last_run"]["status"], "·", state["last_run"]["rows_new"], "new logs")
print("Schedule:", state["schedule_text"], "· next run:", state["next_run"])
print("Open alerts:", state["open_alerts"])

Acknowledge the open alerts, as the button on the Alerts tab does, then stop the app:

In [ ]:
print(app_call("POST", "/api/alerts/all/ack"))
app.terminate()
_ = app.wait()

## Keep it running

This shows the entry that would start the app when you log in. Nothing is installed
until you add `--install` (`--remove` takes it out again). The app reads the API key from
a `.env` file in the monitor folder or the folder above it.

In [ ]:
!{AMYGDA_RISK} service {MON_DIR} --port {PORT}

## Status, and the scores in the database

In [ ]:
!{AMYGDA_RISK} status {MON_DIR}

In [ ]:
con = sqlite3.connect(MON_DIR / "monitor.db")
display(pd.read_sql_query(
    """SELECT asset_id, ROUND(AVG(operational_risk), 1) AS mean,
              ROUND(MAX(operational_risk), 1) AS highest
       FROM scores GROUP BY asset_id ORDER BY highest DESC""", con))
con.close()

## Clean up (optional)

Deletes the deployment and the monitor folder.

In [ ]:
DELETE = False   # ← set True to delete
if DELETE:
    client.delete_deployment(API_KEY, Monitor(MON_DIR).deployment["deployment_id"])
    shutil.rmtree(MON_DIR, ignore_errors=True)
    SPEC_FILE.unlink(missing_ok=True)